# Post-hoc gate analysis

Evaluation only. Requires the actual seed-42 checkpoint/results archive. Review checkpoint path selection before execution.

Source: `graindiscoloration (2).ipynb`, zero-based cell 47.

Run in a fresh Kaggle GPU session with the appropriate dataset attached. Review the configuration near the top; original dataset/output paths are retained. Do not Run All across the repository.


In [ ]:
"""
Standalone post-hoc gate analysis for the trained six-channel proposed model.

Add the original GD dataset and the downloaded seed-42 result ZIP/checkpoint
as Kaggle inputs. Then paste this complete file into one new Kaggle cell.
The code rebuilds the test loader and model, loads the checkpoint, and runs
gate analysis without retraining.
"""

from pathlib import Path
import csv
import json
import math
import shutil
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import DataLoader, Dataset


# --------------------------------------------------------------------------
# 1. Standalone setup: dataset, checkpoint, loader and model
# --------------------------------------------------------------------------

WORK_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
OUT_DIR = WORK_DIR / "six_channel_gate_analysis_seed42"
OUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_SIZE = (896, 672)  # width, height
RGB_CHANNELS = 3
SPECTRAL_CHANNELS = 3
TOTAL_INPUT_CHANNELS = 6
BRANCH_BASE = 28
PREFERRED_GN_GROUPS = 8
MODALITY_DROPOUT_PROBABILITY = 0.0
MASK_POSITIVE_RGB = np.array([240, 128, 128], dtype=np.int16)

DEFAULT_MEAN = np.array(
    [
        0.41819797, 0.46573568, 0.25167707,
        0.21822916, 0.16206901, 0.23585168,
    ],
    dtype=np.float32,
)
DEFAULT_STD = np.array(
    [
        0.15404519, 0.15691774, 0.13375558,
        0.07147331, 0.05124885, 0.04640943,
    ],
    dtype=np.float32,
)

try:
    BILINEAR = Image.Resampling.BILINEAR
    NEAREST = Image.Resampling.NEAREST
except AttributeError:
    BILINEAR = Image.BILINEAR
    NEAREST = Image.NEAREST


def read_list(path):
    with open(path, "r", encoding="utf-8") as file:
        return [line.strip() for line in file if line.strip()]


def valid_dataset_root(path):
    required = [
        path / "RGB",
        path / "MS",
        path / "Mark",
        path / "train_list.txt",
        path / "val_list.txt",
        path / "test_list.txt",
    ]
    if not all(item.exists() for item in required):
        return False
    return (
        len(read_list(path / "train_list.txt")) == 280
        and len(read_list(path / "val_list.txt")) == 60
        and len(read_list(path / "test_list.txt")) == 60
    )


def find_dataset_root():
    search_roots = [Path("/kaggle/input"), Path("/content")]
    candidates = []
    for search_root in search_roots:
        if not search_root.exists():
            continue
        for split_file in search_root.rglob("test_list.txt"):
            if valid_dataset_root(split_file.parent):
                candidates.append(split_file.parent)
    if not candidates:
        raise FileNotFoundError(
            "Could not find the GD dataset with RGB, MS, Mark and the "
            "280/60/60 split files. Add it as a Kaggle input."
        )
    candidates.sort(
        key=lambda path: (
            "gd-rgbms" not in str(path).lower(),
            len(str(path)),
        )
    )
    return candidates[0]


CHECKPOINT_FILENAME = "best_v3_1_without_ndre_seed42.pt"


def find_or_extract_checkpoint():
    search_roots = [Path("/kaggle/input"), Path("/content"), WORK_DIR]

    direct_candidates = []
    for search_root in search_roots:
        if search_root.exists():
            direct_candidates.extend(search_root.rglob(CHECKPOINT_FILENAME))
    if direct_candidates:
        direct_candidates.sort(key=lambda path: len(str(path)))
        return direct_candidates[0]

    zip_candidates = []
    for search_root in search_roots[:2]:
        if search_root.exists():
            zip_candidates.extend(search_root.rglob("*.zip"))

    extraction_dir = WORK_DIR / "extracted_six_channel_checkpoint"
    extraction_dir.mkdir(parents=True, exist_ok=True)

    for archive in zip_candidates:
        try:
            with zipfile.ZipFile(archive, "r") as zip_file:
                matching_members = [
                    name
                    for name in zip_file.namelist()
                    if Path(name).name == CHECKPOINT_FILENAME
                ]
                if not matching_members:
                    continue
                member = matching_members[0]
                zip_file.extract(member, extraction_dir)
                extracted_path = extraction_dir / member
                if extracted_path.exists():
                    return extracted_path
        except zipfile.BadZipFile:
            continue

    raise FileNotFoundError(
        f"Could not find {CHECKPOINT_FILENAME}. Upload the downloaded "
        "six-channel seed-42 result ZIP or the .pt checkpoint as a "
        "Kaggle input."
    )


def safe_load(path, map_location):
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BASE_DIR = find_dataset_root()
checkpoint_path = find_or_extract_checkpoint()
checkpoint = safe_load(checkpoint_path, device)

MEAN = np.asarray(checkpoint.get("mean", DEFAULT_MEAN), dtype=np.float32)
STD = np.asarray(checkpoint.get("std", DEFAULT_STD), dtype=np.float32)
BRANCH_BASE = int(checkpoint.get("branch_base", BRANCH_BASE))

if MEAN.shape != (6,) or STD.shape != (6,):
    raise ValueError("The checkpoint is not the expected six-channel model.")

rgb_dir = BASE_DIR / "RGB"
ms_dir = BASE_DIR / "MS"
mark_dir = BASE_DIR / "Mark"
test_names = read_list(BASE_DIR / "test_list.txt")


def find_path(directory, stem, extensions):
    for extension in extensions:
        path = directory / f"{stem}{extension}"
        if path.exists():
            return path
    raise FileNotFoundError(f"Could not find {stem} in {directory}")


def binary_mask(image):
    array = np.asarray(image.convert("RGB"), dtype=np.int16)
    return (
        np.abs(array - MASK_POSITIVE_RGB).sum(axis=2) < 80
    ).astype(np.float32)


class GateTestDataset(Dataset):
    def __init__(self, names):
        self.names = list(names)

    def __len__(self):
        return len(self.names)

    def __getitem__(self, index):
        stem = Path(self.names[index]).stem
        rgb_path = find_path(
            rgb_dir, stem, [".JPG", ".jpg", ".JPEG", ".jpeg"]
        )
        ms_path = find_path(ms_dir, stem, [".PNG", ".png"])
        mask_path = find_path(mark_dir, stem, [".png", ".PNG"])

        with Image.open(rgb_path) as item:
            rgb_image = item.convert("RGB").resize(TARGET_SIZE, BILINEAR)
        with Image.open(ms_path) as item:
            ms_image = item.convert("RGB").resize(TARGET_SIZE, BILINEAR)
        with Image.open(mask_path) as item:
            mask_image = item.convert("RGB").resize(TARGET_SIZE, NEAREST)

        rgb = np.asarray(rgb_image, dtype=np.float32).copy() / 255.0
        spectral = np.asarray(ms_image, dtype=np.float32).copy() / 255.0
        image = np.concatenate([rgb, spectral], axis=2)
        image = ((image - MEAN) / STD).astype(np.float32)
        mask = binary_mask(mask_image)

        return (
            torch.from_numpy(image.transpose(2, 0, 1)).float(),
            torch.from_numpy(mask[None]).float(),
            stem,
        )


test_loader = DataLoader(
    GateTestDataset(test_names),
    batch_size=1,
    shuffle=False,
    num_workers=0,
    pin_memory=device.type == "cuda",
)


def group_norm(channels):
    groups = min(PREFERRED_GN_GROUPS, channels)
    while channels % groups:
        groups -= 1
    return nn.GroupNorm(groups, channels, eps=1e-5, affine=True)


class SCSE(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        hidden = max(channels // reduction, 1)
        self.channel = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(channels, hidden, 1),
            nn.SiLU(inplace=True),
            nn.Conv2d(hidden, channels, 1),
            nn.Sigmoid(),
        )
        self.spatial = nn.Sequential(
            nn.Conv2d(channels, 1, 1), nn.Sigmoid()
        )

    def forward(self, x):
        return x * self.channel(x) + x * self.spatial(x)


class ResBlock(nn.Module):
    def __init__(self, in_channels, out_channels, dropout=0.0):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False),
            group_norm(out_channels),
            nn.SiLU(inplace=True),
            nn.Dropout2d(dropout) if dropout else nn.Identity(),
            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
            group_norm(out_channels),
        )
        self.skip = (
            nn.Identity()
            if in_channels == out_channels
            else nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, bias=False),
                group_norm(out_channels),
            )
        )
        self.attention = SCSE(out_channels)

    def forward(self, x):
        return self.attention(
            F.silu(self.block(x) + self.skip(x), inplace=True)
        )


class SoftGatedFusion(nn.Module):
    def __init__(self, channels):
        super().__init__()
        hidden = max(channels // 8, 8)
        self.gate = nn.Sequential(
            nn.Conv2d(channels * 2, hidden, 1, bias=False),
            group_norm(hidden),
            nn.SiLU(inplace=True),
            nn.Conv2d(hidden, 2, 1),
        )
        self.refine = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1, bias=False),
            group_norm(channels),
            nn.SiLU(inplace=True),
        )

    def forward(self, rgb_feature, spectral_feature):
        weights = torch.softmax(
            self.gate(torch.cat([rgb_feature, spectral_feature], dim=1)),
            dim=1,
        )
        fused = (
            weights[:, 0:1] * rgb_feature
            + weights[:, 1:2] * spectral_feature
        )
        return self.refine(fused), weights


class UpBlock(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels, dropout=0.0):
        super().__init__()
        self.conv = ResBlock(
            in_channels + skip_channels, out_channels, dropout
        )

    def forward(self, x, skip):
        x = F.interpolate(
            x, size=skip.shape[-2:], mode="bilinear", align_corners=False
        )
        return self.conv(torch.cat([x, skip], dim=1))


class AuxiliarySupervisedGatedUNetNoNDRE(nn.Module):
    def __init__(self, base=28):
        super().__init__()
        c1, c2, c3, c4 = [base, base * 2, base * 4, base * 8]
        self.pool = nn.MaxPool2d(2)
        self.rgb1 = ResBlock(3, c1)
        self.spec1 = ResBlock(3, c1)
        self.rgb2, self.spec2 = ResBlock(c1, c2), ResBlock(c1, c2)
        self.rgb3, self.spec3 = (
            ResBlock(c2, c3, 0.05),
            ResBlock(c2, c3, 0.05),
        )
        self.rgb4, self.spec4 = (
            ResBlock(c3, c4, 0.10),
            ResBlock(c3, c4, 0.10),
        )
        self.fuse1 = SoftGatedFusion(c1)
        self.fuse2 = SoftGatedFusion(c2)
        self.fuse3 = SoftGatedFusion(c3)
        self.fuse4 = SoftGatedFusion(c4)
        self.rgb_auxiliary_head = nn.Conv2d(c4, 1, 1)
        self.spectral_auxiliary_head = nn.Conv2d(c4, 1, 1)
        self.center = ResBlock(c4, c4 * 2, 0.10)
        self.up4 = UpBlock(c4 * 2, c4, c4, 0.10)
        self.up3 = UpBlock(c4, c3, c3, 0.05)
        self.up2 = UpBlock(c3, c2, c2)
        self.up1 = UpBlock(c2, c1, c1)
        self.output = nn.Conv2d(c1, 1, 1)

    def forward(self, x, return_gates=False):
        rgb, spectral = x[:, :3], x[:, 3:]
        r1, s1 = self.rgb1(rgb), self.spec1(spectral)
        f1, g1 = self.fuse1(r1, s1)
        r2 = self.rgb2(self.pool(r1 + f1))
        s2 = self.spec2(self.pool(s1 + f1))
        f2, g2 = self.fuse2(r2, s2)
        r3 = self.rgb3(self.pool(r2 + f2))
        s3 = self.spec3(self.pool(s2 + f2))
        f3, g3 = self.fuse3(r3, s3)
        r4 = self.rgb4(self.pool(r3 + f3))
        s4 = self.spec4(self.pool(s3 + f3))
        f4, g4 = self.fuse4(r4, s4)
        center = self.center(self.pool(f4))
        d4 = self.up4(center, f4)
        d3 = self.up3(d4, f3)
        d2 = self.up2(d3, f2)
        d1 = self.up1(d2, f1)
        logits = self.output(d1)
        gates = [g1, g2, g3, g4]
        return (logits, gates) if return_gates else logits


model = AuxiliarySupervisedGatedUNetNoNDRE(BRANCH_BASE).to(device)
model.load_state_dict(checkpoint["model"], strict=True)
model.eval()

print("Dataset    :", BASE_DIR)
print("Checkpoint :", checkpoint_path)
print("Device     :", device)
print("Test images:", len(test_names))

GATE_THRESHOLD = float(
    checkpoint.get("primary_no_tta_test_metrics", {}).get("threshold", 0.60)
)
MAX_VISUAL_SAMPLES = 4

gate_output_dir = Path(OUT_DIR) / "posthoc_gate_analysis"
gate_output_dir.mkdir(parents=True, exist_ok=True)

model.eval()
mean_vector = np.asarray(MEAN, dtype=np.float32).reshape(1, 1, -1)
std_vector = np.asarray(STD, dtype=np.float32).reshape(1, 1, -1)


def write_csv(path, rows):
    if not rows:
        return
    with open(path, "w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)


class RunningMoments:
    """Pixel-level descriptive mean and population SD without storing maps."""

    def __init__(self):
        self.count = 0
        self.total = 0.0
        self.total_squared = 0.0

    def update(self, values):
        values = values.detach().double()
        if values.numel() == 0:
            return
        self.count += values.numel()
        self.total += values.sum().item()
        self.total_squared += (values * values).sum().item()

    def result(self):
        if self.count == 0:
            return float("nan"), float("nan")
        mean = self.total / self.count
        variance = max(self.total_squared / self.count - mean * mean, 0.0)
        return mean, math.sqrt(variance)


regions = ("overall", "gd", "background")
modalities = ("rgb", "spectral")
moments = {
    scale: {
        region: {modality: RunningMoments() for modality in modalities}
        for region in regions
    }
    for scale in range(1, 5)
}

per_image_rows = []
complementarity_error = 0.0


# --------------------------------------------------------------------------
# 2. Analyse all test images in the original orientation only
# --------------------------------------------------------------------------

with torch.inference_mode():
    for images, masks, stems in test_loader:
        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        logits, gates = model(images, return_gates=True)
        probabilities = torch.sigmoid(logits)
        predictions = probabilities >= GATE_THRESHOLD
        gt = masks >= 0.5

        intersection = (predictions & gt).sum().item()
        prediction_sum = predictions.sum().item()
        target_sum = gt.sum().item()
        image_dice = (2.0 * intersection + 1e-7) / (
            prediction_sum + target_sum + 1e-7
        )

        row = {
            "stem": stems[0],
            "gd_pixel_fraction": float(gt.float().mean().item()),
            "no_tta_dice_at_0.60": float(image_dice),
        }

        for scale, gate in enumerate(gates, start=1):
            complementarity_error = max(
                complementarity_error,
                float((gate.sum(dim=1) - 1.0).abs().max().item()),
            )

            # Upsample to mask resolution for pixel-aligned GD/background analysis.
            gate_up = F.interpolate(
                gate,
                size=masks.shape[-2:],
                mode="bilinear",
                align_corners=False,
            )
            rgb_weight = gate_up[:, 0:1]
            spectral_weight = gate_up[:, 1:2]

            gd_region = gt
            background_region = ~gt

            moments[scale]["overall"]["rgb"].update(rgb_weight)
            moments[scale]["overall"]["spectral"].update(spectral_weight)
            moments[scale]["gd"]["rgb"].update(rgb_weight[gd_region])
            moments[scale]["gd"]["spectral"].update(
                spectral_weight[gd_region]
            )
            moments[scale]["background"]["rgb"].update(
                rgb_weight[background_region]
            )
            moments[scale]["background"]["spectral"].update(
                spectral_weight[background_region]
            )

            row[f"scale{scale}_rgb_mean"] = float(rgb_weight.mean().item())
            row[f"scale{scale}_rgb_spatial_sd"] = float(
                rgb_weight.std(unbiased=False).item()
            )
            row[f"scale{scale}_spectral_mean"] = float(
                spectral_weight.mean().item()
            )
            row[f"scale{scale}_spectral_spatial_sd"] = float(
                spectral_weight.std(unbiased=False).item()
            )
            row[f"scale{scale}_rgb_gd_mean"] = (
                float(rgb_weight[gd_region].mean().item())
                if gd_region.any()
                else float("nan")
            )
            row[f"scale{scale}_spectral_gd_mean"] = (
                float(spectral_weight[gd_region].mean().item())
                if gd_region.any()
                else float("nan")
            )
            row[f"scale{scale}_rgb_background_mean"] = float(
                rgb_weight[background_region].mean().item()
            )
            row[f"scale{scale}_spectral_background_mean"] = float(
                spectral_weight[background_region].mean().item()
            )

        per_image_rows.append(row)


# --------------------------------------------------------------------------
# 3. Create paper-ready summary tables
# --------------------------------------------------------------------------

region_summary_rows = []
scale_summary_rows = []

for scale in range(1, 5):
    for region in regions:
        rgb_mean, rgb_sd = moments[scale][region]["rgb"].result()
        spectral_mean, spectral_sd = moments[scale][region]["spectral"].result()
        region_summary_rows.append(
            {
                "scale": scale,
                "region": region,
                "pixel_count": moments[scale][region]["rgb"].count,
                "rgb_weight_mean": rgb_mean,
                "rgb_weight_sd": rgb_sd,
                "spectral_weight_mean": spectral_mean,
                "spectral_weight_sd": spectral_sd,
                "spectral_minus_rgb": spectral_mean - rgb_mean,
            }
        )

    overall = next(
        item
        for item in region_summary_rows
        if item["scale"] == scale and item["region"] == "overall"
    )
    gd = next(
        item
        for item in region_summary_rows
        if item["scale"] == scale and item["region"] == "gd"
    )
    background = next(
        item
        for item in region_summary_rows
        if item["scale"] == scale and item["region"] == "background"
    )

    image_spatial_sds = np.asarray(
        [
            item[f"scale{scale}_spectral_spatial_sd"]
            for item in per_image_rows
        ],
        dtype=np.float64,
    )
    scale_summary_rows.append(
        {
            "scale": scale,
            "rgb_overall_mean": overall["rgb_weight_mean"],
            "spectral_overall_mean": overall["spectral_weight_mean"],
            "spectral_spatial_sd_mean_across_images": float(
                image_spatial_sds.mean()
            ),
            "spectral_spatial_sd_across_images": float(
                image_spatial_sds.std(ddof=1)
            ) if len(image_spatial_sds) > 1 else 0.0,
            "spectral_gd_mean": gd["spectral_weight_mean"],
            "spectral_background_mean": background["spectral_weight_mean"],
            "spectral_gd_minus_background": (
                gd["spectral_weight_mean"]
                - background["spectral_weight_mean"]
            ),
        }
    )

write_csv(gate_output_dir / "per_image_gate_statistics.csv", per_image_rows)
write_csv(gate_output_dir / "gate_region_summary.csv", region_summary_rows)
write_csv(gate_output_dir / "gate_scale_summary_for_paper.csv", scale_summary_rows)


# --------------------------------------------------------------------------
# 4. Select four representative positive-GD images by lesion fraction
# --------------------------------------------------------------------------

positive_rows = [
    item for item in per_image_rows if item["gd_pixel_fraction"] > 0
]
positive_rows.sort(key=lambda item: item["gd_pixel_fraction"])

if not positive_rows:
    raise RuntimeError("No positive-GD test image was found.")

quantiles = np.linspace(0.10, 0.90, min(MAX_VISUAL_SAMPLES, len(positive_rows)))
selected_stems = []
for quantile in quantiles:
    index = int(round(quantile * (len(positive_rows) - 1)))
    stem = positive_rows[index]["stem"]
    if stem not in selected_stems:
        selected_stems.append(stem)

# Fill any duplicate quantile selections with unused positive samples.
for item in positive_rows:
    if len(selected_stems) >= min(MAX_VISUAL_SAMPLES, len(positive_rows)):
        break
    if item["stem"] not in selected_stems:
        selected_stems.append(item["stem"])


# --------------------------------------------------------------------------
# 5. Re-run only selected images and draw fixed-range spectral gate maps
# --------------------------------------------------------------------------

visual_records = []
selected_set = set(selected_stems)

with torch.inference_mode():
    for images, masks, stems in test_loader:
        stem = stems[0]
        if stem not in selected_set:
            continue

        images_device = images.to(device, non_blocking=True)
        logits, gates = model(images_device, return_gates=True)
        probability = torch.sigmoid(logits)[0, 0].cpu().numpy()
        prediction = probability >= GATE_THRESHOLD
        gt_mask = masks[0, 0].numpy() >= 0.5

        normalized_image = images[0].permute(1, 2, 0).numpy()
        original_image = normalized_image * std_vector + mean_vector
        rgb_image = np.clip(original_image[..., :3], 0.0, 1.0)

        spectral_maps = []
        for gate in gates:
            spectral_map = F.interpolate(
                gate[:, 1:2],
                size=masks.shape[-2:],
                mode="bilinear",
                align_corners=False,
            )[0, 0].cpu().numpy()
            spectral_maps.append(spectral_map)

        metrics_row = next(
            item for item in per_image_rows if item["stem"] == stem
        )
        visual_records.append(
            {
                "stem": stem,
                "rgb": rgb_image,
                "gt": gt_mask,
                "prediction": prediction,
                "spectral_maps": spectral_maps,
                "gd_fraction": metrics_row["gd_pixel_fraction"],
                "dice": metrics_row["no_tta_dice_at_0.60"],
            }
        )

visual_records.sort(key=lambda item: item["gd_fraction"])

column_titles = [
    "RGB",
    "Ground truth",
    "Prediction",
    "Spectral gate S1",
    "Spectral gate S2",
    "Spectral gate S3",
    "Spectral gate S4",
]

figure, axes = plt.subplots(
    len(visual_records),
    len(column_titles),
    figsize=(21, 3.4 * len(visual_records)),
    squeeze=False,
)

last_gate_image = None
for row_index, record in enumerate(visual_records):
    panels = [record["rgb"], record["gt"], record["prediction"]]
    for column_index, panel in enumerate(panels):
        cmap = None if column_index == 0 else "gray"
        axes[row_index, column_index].imshow(panel, cmap=cmap)

    for scale_index, gate_map in enumerate(record["spectral_maps"]):
        last_gate_image = axes[row_index, scale_index + 3].imshow(
            gate_map,
            cmap="viridis",
            vmin=0.0,
            vmax=1.0,
        )

    axes[row_index, 0].set_ylabel(
        f"{record['stem']}\n"
        f"GD={100 * record['gd_fraction']:.2f}% | "
        f"Dice={record['dice']:.3f}",
        fontsize=9,
    )

    for column_index, title in enumerate(column_titles):
        if row_index == 0:
            axes[row_index, column_index].set_title(title, fontsize=10)
        axes[row_index, column_index].set_xticks([])
        axes[row_index, column_index].set_yticks([])

figure.suptitle(
    "Learned spectral-modality weights across four encoder scales",
    fontsize=14,
    y=0.995,
)
if last_gate_image is not None:
    colorbar = figure.colorbar(
        last_gate_image,
        ax=axes[:, 3:].ravel().tolist(),
        fraction=0.012,
        pad=0.01,
    )
    colorbar.set_label("Spectral weight (RGB weight = 1 - spectral weight)")

figure.subplots_adjust(left=0.08, right=0.94, top=0.95, bottom=0.03,
                       wspace=0.04, hspace=0.12)
figure_path = gate_output_dir / "representative_spectral_gate_maps.png"
figure.savefig(figure_path, dpi=300, bbox_inches="tight")
plt.show()
plt.close(figure)


# --------------------------------------------------------------------------
# 6. Save metadata and make a small downloadable archive
# --------------------------------------------------------------------------

metadata = {
    "model": "Auxiliary-Supervised Spatially Gated Multi-Scale Fusion Network",
    "input_channels": ["R", "G", "B", "Green", "Red-Edge", "NIR"],
    "seed": 42,
    "split": "test",
    "inference": "original orientation, no TTA",
    "prediction_threshold": GATE_THRESHOLD,
    "gate_definition": (
        "Two softmax weights at every pixel and scale; RGB and spectral "
        "weights sum to one."
    ),
    "region_definition": "GD region uses the ground-truth binary mask.",
    "selected_visual_stems": selected_stems,
    "maximum_gate_sum_error": complementarity_error,
    "interpretation_note": (
        "Spatial SD and GD-background differences are descriptive. "
        "They are not statistical significance tests."
    ),
}
with open(gate_output_dir / "gate_analysis_metadata.json", "w", encoding="utf-8") as file:
    json.dump(metadata, file, indent=2)

archive_base = gate_output_dir.parent / "six_channel_gate_analysis_seed42"
archive_path = shutil.make_archive(str(archive_base), "zip", gate_output_dir)

print("\nGATE SCALE SUMMARY")
for item in scale_summary_rows:
    print(
        f"S{item['scale']}: "
        f"RGB={item['rgb_overall_mean']:.4f}, "
        f"Spectral={item['spectral_overall_mean']:.4f}, "
        f"Mean spatial SD={item['spectral_spatial_sd_mean_across_images']:.4f}, "
        f"Spectral GD-BG={item['spectral_gd_minus_background']:+.4f}"
    )

print(f"\nMaximum |RGB + spectral - 1|: {complementarity_error:.3e}")
print("Figure :", figure_path)
print("CSV    :", gate_output_dir / "gate_scale_summary_for_paper.csv")
print("ZIP    :", archive_path)

try:
    from IPython.display import FileLink, display
    display(FileLink(archive_path))
except Exception:
    pass